# Week 11: Advanced Specialized Machine Learning
## Notebook 07: Tabular Synthetic Data Generation: Mode-Specific Normalization & CTGAN

### 1. Conceptual Intuition & Theoretical Background
Generating synthetic tabular data using standard GANs fails because tabular data possesses:
1. Mixed data types (discrete categorical + continuous numerical).
2. **Multi-modal continuous distributions** (e.g. salary or transaction amounts having multiple cluster peaks).
3. Highly skewed or sparse categories.

**CTGAN (Conditional Tabular GAN)** resolves this using:
1. **Mode-Specific Normalization:** Continuous column $C_i$ is modeled using a Variational Gaussian Mixture Model (VGM) with $K$ components:
$$P(c_i) = \sum_{k=1}^{K} w_k \mathcal{N}(\mu_k, \sigma_k^2)$$
Each scalar value is encoded as a representation pair:
- A one-hot vector $\beta \in \{0, 1\}^K$ identifying the closest Gaussian component mode.
- A normalized scalar $\alpha = \frac{c_i - \mu_k}{4\sigma_k} \in [-1, 1]$.

2. **Conditional Generator:** Samples discrete categories conditionally to ensure rare categories are sampled frequently during training.

---

### 2. Scratch Python Implementation: Mode-Specific Normalization

In [ ]:
import numpy as np

class ModeSpecificNormalizer:
    def __init__(self, n_modes=3):
        self.n_modes = n_modes
        self.means = []
        self.stds = []
        self.weights = []
        
    def fit(self, values):
        # Scratch 1D Gaussian Mixture approximation using quantile clustering
        quantiles = np.linspace(0, 100, self.n_modes + 1)
        bins = np.percentile(values, quantiles)
        
        self.means = []
        self.stds = []
        for i in range(self.n_modes):
            mask = (values >= bins[i]) & (values <= bins[i+1])
            subset = values[mask]
            if len(subset) > 1:
                self.means.append(np.mean(subset))
                self.stds.append(max(np.std(subset), 1e-4))
            else:
                self.means.append(bins[i])
                self.stds.append(1.0)
                
        self.means = np.array(self.means)
        self.stds = np.array(self.stds)
        return self
        
    def transform(self, values):
        # Encode each scalar as (scalar alpha, one-hot beta)
        alphas = []
        betas = []
        for v in values:
            # Find closest mode
            mode_idx = np.argmin(np.abs(v - self.means))
            # Normalized scalar
            alpha = (v - self.means[mode_idx]) / (4.0 * self.stds[mode_idx])
            beta = np.zeros(self.n_modes)
            beta[mode_idx] = 1.0
            alphas.append(alpha)
            betas.append(beta)
            
        return np.array(alphas), np.array(betas)
        
    def inverse_transform(self, alphas, betas):
        recovered = []
        for a, b in zip(alphas, betas):
            mode_idx = np.argmax(b)
            v = a * (4.0 * self.stds[mode_idx]) + self.means[mode_idx]
            recovered.append(v)
        return np.array(recovered)

# Bimodal synthetic distribution
np.random.seed(42)
cluster_1 = np.random.normal(loc=20.0, scale=2.0, size=500)
cluster_2 = np.random.normal(loc=85.0, scale=5.0, size=500)
raw_features = np.concatenate([cluster_1, cluster_2])

normalizer = ModeSpecificNormalizer(n_modes=2).fit(raw_features)
alphas, betas = normalizer.transform(raw_features)
reconstructed = normalizer.inverse_transform(alphas, betas)

reconstruction_mae = np.mean(np.abs(raw_features - reconstructed))
print("--- CTGAN Mode-Specific Normalization ---")
print(f"Original Feature Min/Max:       [{raw_features.min():.1f}, {raw_features.max():.1f}]")
print(f"Encoded Alpha Scalar Range:     [{alphas.min():.3f}, {alphas.max():.3f}]")
print(f"Reconstruction Absolute Error:  {reconstruction_mae:.6f} (Exact inversion!)")

### 3. Step-by-Step Code Walkthrough
1. **Multimodal Disentanglement**: Partitions arbitrary distributions into bounded normal components.
2. **Representation Invariance**: Enables deep neural networks with $\tanh$ output layers to generate bounded continuous values without mode saturation.
3. **Exact Recovery**: Inverse transformation reconstructs original domain units with zero numerical degradation.

---

### 4. Industry Applications & Trade-offs
- **Healthcare & Banking Privacy:** Synthesizing patient health records and financial transactions for GDPR-compliant model benchmarking.
- **Trade-off:** Training conditional GANs on wide tables ($>100$ columns) requires careful tuning of WGAN-GP gradient penalty coefficients.

---

### 5. Top Interview Questions & Answers
**Q1: Why do standard GANs suffer from mode collapse when trained on tabular data?**
> **Answer:** Tabular features often have multi-modal, highly non-Gaussian distributions (e.g. many zeros followed by a long tail). Standard min-max normalization forces the generator to focus on dominant modes, completely ignoring sparse regions.

**Q2: How does the Conditional Generator in CTGAN address class imbalance?**
> **Answer:** It samples conditional vectors $c$ by uniformly sampling discrete categories (giving equal probability to rare and frequent categories) and conditions the generator to synthesize samples matching that specific category.